# 04 — Beacon Periodicity Analysis

**Goal:** find candidate C2 beacons by analyzing inter-arrival times per (src, dst, port).

We use the same shape as the `Gigamon_Beacon_Periodicity_Hunt` MCP tool — but with full visual drill-down.

## Setup

Authenticate with Azure CLI (`az login`) and load workspace coordinates from `.env`.

In [ ]:
import os, datetime as dt
import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

load_dotenv()
WORKSPACE_ID = os.environ['WORKSPACE_ID']
HOURS = int(os.environ.get('TIMERANGE_HOURS', '24'))
client = LogsQueryClient(DefaultAzureCredential())
TIMESPAN = dt.timedelta(hours=HOURS)

def kql(q: str) -> pd.DataFrame:
    """Run a KQL query and return the first table as a DataFrame."""
    r = client.query_workspace(WORKSPACE_ID, q, timespan=TIMESPAN)
    if r.status != LogsQueryStatus.SUCCESS:
        raise RuntimeError(r.partial_error)
    t = r.tables[0]
    return pd.DataFrame(t.rows, columns=[c for c in t.columns])

## Pull candidate flow streams

In [ ]:
raw = kql('''
GigamonCcfMcpDemo_CL
| where isnotempty(src_ip) and isnotempty(dst_ip) and isnotempty(dst_port)
| project TimeGenerated, src_ip, dst_ip, dst_port, app_name, total_bytes
| order by src_ip, dst_ip, dst_port, TimeGenerated asc
''')
raw['TimeGenerated'] = pd.to_datetime(raw['TimeGenerated'])
len(raw)

## Compute inter-arrival gaps per (src, dst, port)

In [ ]:
raw['GapSec'] = (raw
                .groupby(['src_ip','dst_ip','dst_port'])['TimeGenerated']
                .diff()
                .dt.total_seconds())
gaps = raw.dropna(subset=['GapSec'])
gaps = gaps[(gaps['GapSec'] > 0) & (gaps['GapSec'] < 14400)]
gaps.head()

## Per-pair periodicity metrics

In [ ]:
import numpy as np
def iqr(s):
    return s.quantile(0.75) - s.quantile(0.25)
agg = gaps.groupby(['src_ip','dst_ip','dst_port']).agg(
    samples=('GapSec','count'),
    median_gap=('GapSec','median'),
    std_gap=('GapSec','std'),
    iqr_gap=('GapSec', iqr),
).reset_index()
agg['jitter'] = agg['std_gap'] / agg['median_gap']
agg['iqr_ratio'] = agg['iqr_gap'] / agg['median_gap']
beacons = agg[(agg['samples'] >= 5) & (agg['jitter'] < 0.25) & (agg['iqr_ratio'] < 0.3) & (agg['median_gap'] >= 15)]
beacons.sort_values('samples', ascending=False)

## Plot inter-arrival histogram for the strongest candidate

In [ ]:
import matplotlib.pyplot as plt
if not beacons.empty:
    top = beacons.sort_values('samples', ascending=False).iloc[0]
    sub = gaps[(gaps['src_ip']==top['src_ip']) & (gaps['dst_ip']==top['dst_ip']) & (gaps['dst_port']==top['dst_port'])]
    plt.figure(figsize=(10,4))
    plt.hist(sub['GapSec'], bins=40, color='#FFD74B', edgecolor='#742DDC')
    plt.title(f'Beacon candidate {top.src_ip} → {top.dst_ip}:{top.dst_port}  (median {top.median_gap:.1f}s)')
    plt.xlabel('Inter-arrival gap (s)'); plt.ylabel('count')
    plt.show()
else:
    print('No beacons in current window. Consider widening TIMERANGE_HOURS.')